# Value factors: how to obtain and compute them (practical)

This notebook builds a **single-row panel of common value / yield metrics** from **market price**, **shares**, and **financial statements** (income, balance sheet, cash flow), plus a few fields from `yfinance`’s `info` for forward-looking ratios where Yahoo exposes them.

**Parameters** are in the configuration cell below (ticker, optional peer list for relative forward P/E).

## How to obtain the data (2026)

| Use case | Source | Notes | Cost |
|----------|--------|-------|------|
| Strong fundamentals + ratios API | **Financial Modeling Prep (FMP)** | Good for production-style factor pipelines; many ratios precomputed | Paid (free tier) |
| Fundamentals + quality price feed | **Polygon.io** | Useful when you want vendor prices aligned to fundamentals | Paid |
| Quick experiments | **yfinance** | Free, easy; statement labels and history can be noisy or incomplete | Free |
| Clean standardized statements | **SimFin** | Research-friendly standardized line items | Free / paid |
| Institutional PIT / consensus | **Bloomberg, FactSet, Refinitiv / LSEG** | Highest quality, true point-in-time and IBES-style estimates | Very expensive |

**This notebook** implements the **yfinance** path so you can run it without API keys. For live alpha research, plan to move to a provider with **point-in-time (PIT)** fundamentals and a proper **estimates** database for forward sales and sector aggregates.

## Look-ahead bias and TTM

- **Point-in-time**: Yahoo Finance data in `yfinance` reflects **today’s view** of past filings (restatements, mapping). It is **not** suitable for backtests that require “what investors knew on date *t*” unless you switch to a PIT vendor (FMP historical as-reported where offered, SimFin, Polygon, or institutional feeds).
- **TTM (trailing twelve months)**: For flow variables (cash flow, revenue, net income, EBITDA, CapEx), we **sum the last four fiscal quarters** available in the quarterly statements. Columns are sorted by period end date.
- **Stock variables** (total assets, debt, equity, cash on balance sheet): we use the **latest reported quarter** in the balance sheet as the denominator snapshot, while flows use TTM in the numerator—common in screens; document if you need full consistency on a single as-of date from one 10-Q/10-K only.

## Related material in this repo

- Deeper **`yfinance`** API tour: [`02_market_and_fundamental_data/03_data_providers/02_yfinance_demo.ipynb`](../02_market_and_fundamental_data/03_data_providers/02_yfinance_demo.ipynb)
- Broader factor examples and evaluation: [`24_alpha_factor_library/`](../24_alpha_factor_library/) (appendix)

In [30]:
# %pip install yfinance pandas numpy requests

In [31]:
import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
SYMBOL = "TSLA"  # Change to any Yahoo symbol, e.g. "AAPL", "XOM"

# Optional: tickers in the same industry/sector for a crude forward P/E relative metric.
# Leave empty to skip (NaN). This is NOT a full sector consensus—just a peer average from Yahoo `forwardPE`.
FORWARD_PE_PEERS = []  # e.g. ["ORCL", "CRM", "ADBE"] for a software basket

# Optional: set in the environment if you use the FMP skeleton cell at the end
# export FMP_API_KEY=...
import os
FMP_API_KEY = os.getenv("FMP_API_KEY", "r1LMLItOWKfqyVmsj2k95QHUcqP6p9Ba")

## 1. Imports

In [32]:
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf

In [33]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## 2. Statement helpers (regex on line items)

Yahoo statement **row labels vary by company and over time**. Each pattern is a tuple of substrings; **all** must match (case-insensitive) for a row to qualify. We take the **first** matching row in index order.

In [34]:
def _patterns_match(label: Any, patterns: Tuple[str, ...]) -> bool:
    text = str(label).lower()
    return all(p.lower() in text for p in patterns)


def _first_matching_row(df: Optional[pd.DataFrame], patterns: Tuple[str, ...]) -> Optional[pd.Series]:
    if df is None or df.empty:
        return None
    for idx in df.index:
        if _patterns_match(idx, patterns):
            return df.loc[idx]
    return None


def _sorted_period_columns(row: pd.Series) -> List[Any]:
    cols = [c for c in row.index if pd.notna(c)]
    try:
        return sorted(cols, reverse=True)
    except TypeError:
        return cols


def _ttm_sum(df: Optional[pd.DataFrame], patterns: Tuple[str, ...], n: int = 4) -> float:
    row = _first_matching_row(df, patterns)
    if row is None:
        return np.nan
    cols = _sorted_period_columns(row)[:n]
    if not cols:
        return np.nan
    return float(pd.to_numeric(row[cols], errors="coerce").sum())


def _latest_point(df: Optional[pd.DataFrame], patterns: Tuple[str, ...]) -> float:
    row = _first_matching_row(df, patterns)
    if row is None:
        return np.nan
    cols = _sorted_period_columns(row)
    if not cols:
        return np.nan
    return float(pd.to_numeric(row[cols[0]], errors="coerce"))


def _safe_float(x: Any) -> float:
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return np.nan
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def _ttm_diluted_eps(q_fin: pd.DataFrame) -> float:
    """Sum last four quarterly diluted EPS rows when Yahoo exposes them."""
    for patterns in (("diluted", "eps"), ("diluted", "earnings", "share")):
        row = _first_matching_row(q_fin, patterns)
        if row is None:
            continue
        cols = _sorted_period_columns(row)[:4]
        if cols:
            return float(pd.to_numeric(row[cols], errors="coerce").sum())
    return np.nan


def _ttm_eps_from_quarterly_net_income_and_shares(q_inc: pd.DataFrame, q_bs: pd.DataFrame) -> float:
    """Fallback: NI TTM / mean diluted shares over the same quarters (rough)."""
    ni_row = _first_matching_row(q_inc, ("net income",))
    if ni_row is None:
        ni_row = _first_matching_row(q_inc, ("net", "income"))
    sh_row = _first_matching_row(q_bs, ("ordinary", "shares", "number"))
    if sh_row is None:
        sh_row = _first_matching_row(q_bs, ("share", "issued"))
    if ni_row is None:
        return np.nan
    ni_cols = _sorted_period_columns(ni_row)[:4]
    ni_sum = pd.to_numeric(ni_row[ni_cols], errors="coerce").sum()
    if sh_row is not None:
        sh_cols = [c for c in ni_cols if c in sh_row.index]
        if sh_cols:
            sh_mean = pd.to_numeric(sh_row[sh_cols], errors="coerce").mean()
            if sh_mean and not np.isnan(sh_mean) and sh_mean != 0:
                return float(ni_sum / sh_mean)
    return np.nan


def _ebitda_ttm(q_fin: pd.DataFrame, q_cf: pd.DataFrame) -> float:
    v = _ttm_sum(q_fin, ("ebitda",))
    if not np.isnan(v):
        return v
    return _ttm_sum(q_cf, ("ebitda",))


def _fmt_count(n: float) -> str:
    """Shares or other unit counts — commas + scale label."""
    if np.isnan(n):
        return "n/a"
    if abs(n) >= 1e9:
        return f"{n:,.0f}  ({n / 1e9:,.3f} billion)"
    if abs(n) >= 1e6:
        return f"{n:,.0f}  ({n / 1e6:,.2f} million)"
    return f"{n:,.0f}"


def _fmt_usd(n: float) -> str:
    """Dollar amounts — commas + $T / $B label."""
    if np.isnan(n):
        return "n/a"
    if abs(n) >= 1e12:
        return f"${n:,.0f}  (${n / 1e12:,.3f} trillion)"
    if abs(n) >= 1e9:
        return f"${n:,.0f}  (${n / 1e9:,.3f} billion)"
    if abs(n) >= 1e6:
        return f"${n:,.0f}  (${n / 1e6:,.2f} million)"
    return f"${n:,.2f}"

## 3. Load `yfinance` data

In [35]:
ticker = yf.Ticker(SYMBOL)
info: Dict[str, Any] = ticker.info or {}
info

{'address1': '1 Tesla Road',
 'city': 'Austin',
 'state': 'TX',
 'zip': '78725',
 'country': 'United States',
 'phone': '512 516 8177',
 'website': 'https://www.tesla.com',
 'industry': 'Auto Manufacturers',
 'industryKey': 'auto-manufacturers',
 'industryDisp': 'Auto Manufacturers',
 'sector': 'Consumer Cyclical',
 'sectorKey': 'consumer-cyclical',
 'sectorDisp': 'Consumer Cyclical',
 'longBusinessSummary': 'Tesla, Inc. designs, develops, manufactures, leases, and sells electric vehicles, and energy generation and storage systems in the United States, China, and internationally. The company operates in two segments, Automotive; and Energy Generation and Storage. The company offers electric vehicles, as well as sells automotive regulatory credits; and non-warranty maintenance services and collision, automotive insurance services, as well as part sales and retail merchandise sale. It also provides sedans and sport utility vehicles through direct and used vehicle sales, a network of Tesl

In [36]:
q_cf = ticker.quarterly_cashflow
q_cf

,2026-03-31,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31,2024-09-30
Free Cash Flow,1.444000e+09,1.420000e+09,3.990000e+09,1.460000e+08,6.640000e+08,NaN,NaN
Repayment Of Debt,-3.548000e+09,-7.670000e+08,-6.870000e+08,-2.847000e+09,-1.349000e+09,NaN,NaN
Issuance Of Debt,4.331000e+09,1.354000e+09,1.182000e+09,2.425000e+09,6.250000e+08,NaN,NaN
Capital Expenditure,-2.493000e+09,-2.393000e+09,-2.248000e+09,-2.394000e+09,-1.492000e+09,NaN,NaN
End Cash Position,1.765500e+10,1.761600e+10,1.958400e+10,1.673500e+10,1.725000e+10,NaN,NaN
Beginning Cash Position,1.761600e+10,1.958400e+10,1.673500e+10,1.725000e+10,1.703700e+10,NaN,NaN
Effect Of Exchange Rate Changes,-4.700000e+07,3.700000e+07,-1.700000e+07,1.110000e+08,4.000000e+07,NaN,NaN
Changes In Cash,8.600000e+07,-2.005000e+09,2.866000e+09,-6.260000e+08,1.730000e+08,NaN,NaN
Financing Cash Flow,1.172000e+09,7.100000e+08,9.830000e+08,-2.220000e+08,-3.320000e+08,NaN,NaN
Cash Flow From Continuing Financing Activities,1.172000e+09,7.100000e+08,9.830000e+08,-2.220000e+08,-3.320000e+08,NaN,NaN


In [37]:
q_fin = ticker.quarterly_financials
q_fin

,2026-03-31,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31,2024-09-30
Tax Effect Of Unusual Items,0.000000e+00,-4.458086e+07,-6.902000e+07,0.000000e+00,-2.726000e+07,NaN,NaN
Tax Rate For Calcs,3.400000e-01,2.751910e-01,2.900000e-01,2.300000e-01,2.900000e-01,NaN,NaN
Normalized EBITDA,2.430000e+09,3.071000e+09,3.898000e+09,3.068000e+09,2.221000e+09,NaN,NaN
Total Unusual Items,0.000000e+00,-1.620000e+08,-2.380000e+08,0.000000e+00,-9.400000e+07,NaN,NaN
Total Unusual Items Excluding Goodwill,0.000000e+00,-1.620000e+08,-2.380000e+08,0.000000e+00,-9.400000e+07,NaN,NaN
Net Income From Continuing Operation Net Minority Interest,4.770000e+08,8.400000e+08,1.373000e+09,1.172000e+09,4.090000e+08,NaN,NaN
Reconciled Depreciation,1.590000e+09,1.643000e+09,1.625000e+09,1.433000e+09,1.447000e+09,NaN,NaN
Reconciled Cost Of Revenue,1.766700e+10,1.989200e+10,2.304100e+10,1.861800e+10,1.618200e+10,NaN,NaN
EBITDA,2.430000e+09,2.909000e+09,3.660000e+09,3.068000e+09,2.127000e+09,NaN,NaN
EBIT,8.400000e+08,1.266000e+09,2.035000e+09,1.635000e+09,6.800000e+08,NaN,NaN


In [38]:
q_bs = ticker.quarterly_balance_sheet
q_bs


,2026-03-31,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31,2024-09-30
Treasury Shares Number,NaN,NaN,NaN,NaN,NaN,NaN,0.0
Ordinary Shares Number,3.755000e+09,3.751000e+09,3.324000e+09,3.224000e+09,3.220000e+09,NaN,NaN
Share Issued,3.755000e+09,3.751000e+09,3.324000e+09,3.224000e+09,3.220000e+09,NaN,NaN
Total Debt,1.589000e+10,1.471900e+10,1.378800e+10,1.313400e+10,1.312800e+10,NaN,NaN
Tangible Book Value,8.333000e+10,8.074800e+10,7.958200e+10,7.691800e+10,7.426100e+10,NaN,NaN
Invested Capital,9.313500e+10,9.029000e+10,8.743100e+10,8.427000e+10,8.189700e+10,NaN,NaN
Working Capital,3.561000e+10,3.692800e+10,3.336300e+10,3.112500e+10,2.963600e+10,NaN,NaN
Net Tangible Assets,8.333000e+10,8.074800e+10,7.958200e+10,7.691800e+10,7.426100e+10,NaN,NaN
Capital Lease Obligations,6.871000e+09,6.566000e+09,6.327000e+09,6.178000e+09,5.884000e+09,NaN,NaN
Common Stock Equity,8.411600e+10,8.213700e+10,7.997000e+10,7.731400e+10,7.465300e+10,NaN,NaN


### Share price (`info`)

**Current share price** — The market price per share used in yield and market-cap checks. We read it from Yahoo `info` (`currentPrice` or `regularMarketPrice`). If those are missing, we use the **last daily close** from a short `history` pull.

The cell below sets `price` and prints it for `SYMBOL`.


In [39]:

# Price: prefer live quote keys, else last daily close
price = _safe_float(info.get("currentPrice") or info.get("regularMarketPrice"))
if np.isnan(price):
    hist = ticker.history(period="5d")
    if not hist.empty:
        price = float(hist["Close"].iloc[-1])
print(f"Loaded {SYMBOL}: price={price:.4f}" if not np.isnan(price) else f"Loaded {SYMBOL}")

Loaded TSLA: price=443.3000


### Market cap and shares outstanding (`info`)

**1. Shares outstanding** — Total common shares **issued and currently outstanding** (held by institutions, insiders, and the public float). This is the **official** share count from company filings, read from Yahoo as `info['sharesOutstanding']`.

**2. Market capitalization** — What the market thinks the **equity** of the company is worth (most common measure of **company size**):

$$\text{Market Cap} = \text{Share Price} \times \text{Shares Outstanding}$$

The cell below loads both fields from `info`, prints them in readable form, and checks that **price × shares** is close to `marketCap` (small gaps are normal if Yahoo uses a slightly different price or share count timestamp).



In [40]:
shares = _safe_float(info.get("sharesOutstanding"))
market_cap = _safe_float(info.get("marketCap"))

print(f"Shares outstanding:     {_fmt_count(shares)}")
print(f"Market cap:        {_fmt_usd(market_cap)}")


implied_cap = price * shares
diff_pct = 100 * abs(implied_cap - market_cap) / market_cap
print("")
print("Market cap identity (should be close):")
print(f"  price × shares  =  {_fmt_usd(implied_cap)}   @ ${price:,.2f}/share")
print(f"  marketCap       =  {_fmt_usd(market_cap)}   (from info['marketCap'])")
print(f"  gap             =  {_fmt_usd(implied_cap - market_cap)}  ({diff_pct:.2f}%)")


Shares outstanding:     3,755,723,871  (3.756 billion)
Market cap:        $1,664,912,326,656  ($1.665 trillion)

Market cap identity (should be close):
  price × shares  =  $1,664,912,392,014  ($1.665 trillion)   @ $443.30/share
  marketCap       =  $1,664,912,326,656  ($1.665 trillion)   (from info['marketCap'])
  gap             =  $65,358.30  (0.00%)


### Debt (balance sheet)

**1. Short-term debt (current debt)** — Obligations due within **12 months** (e.g. bank loans, current portion of long-term debt, commercial paper, drawn revolvers). Reported under **current liabilities**; must be repaid soon → higher liquidity risk, often higher rates.

**2. Long-term debt** — Due **after one year** (e.g. bonds, multi-year bank loans, convertible notes). Under **non-current liabilities**; more time to repay, usually lower rates than short-term debt.

**3. Total debt** — Sum of **interest-bearing** financial debt (the figure used in valuation):

$$\text{Total Debt} = \text{Short-Term Debt} + \text{Long-Term Debt}$$

Typically **excludes** operating liabilities (accounts payable, accrued expenses, deferred revenue). The cell below reads Yahoo’s `Total Debt` line from `q_bs`, or builds it from long-term + short-term if that total is missing.


In [41]:
total_debt = _latest_point(q_bs, ("total debt",))
print(f"Total Debt:          {_fmt_usd(total_debt)}")

if np.isnan(total_debt):
    ltd = _latest_point(q_bs, ("long term debt",))
    print(f"    Long Term Debt:     {_fmt_usd(ltd)}")
    std = _latest_point(q_bs, ("short", "term", "debt"))
    print(f"    Short Term Debt:    {_fmt_usd(std)}")
    if not (np.isnan(ltd) and np.isnan(std)):
        total_debt = np.nansum([ltd, std])

Total Debt:          $15,890,000,000  ($15.890 billion)


### Cash (balance sheet)

**Cash and cash equivalents** — liquid assets on the balance sheet (bank balances, money-market funds, very short-term instruments). Often under **current assets**.

The cell below reads cash from the latest quarter on `q_bs`, with fallbacks to Yahoo `totalCash` / `totalCashPerShare` × shares if the balance-sheet line is missing.


In [ ]:
cash_bs = _latest_point(q_bs, ("cash and cash equivalents",))
if np.isnan(cash_bs):
    cash_bs = _latest_point(q_bs, ("cash", "equivalents"))

cash_for_ev = cash_bs
if np.isnan(cash_for_ev):
    cash_for_ev = _safe_float(info.get("totalCash"))
if np.isnan(cash_for_ev):
    tcps = _safe_float(info.get("totalCashPerShare"))
    if not np.isnan(tcps) and not np.isnan(shares) and shares > 0:
        cash_for_ev = tcps * shares

print(f"Cash (balance sheet): {_fmt_usd(cash_bs)}")
print(f"Cash (for EV):        {_fmt_usd(cash_for_ev)}")


Cash (balance sheet): $16,603,000,000  ($16.603 billion)
Cash (for EV):        $16,603,000,000  ($16.603 billion)


### Enterprise value

Combines **market cap**, **total debt**, and **cash** (from the cells above) into firm value:

$$\text{Enterprise Value} = \text{Market Cap} + \text{Total Debt} - \text{Cash}$$

We compare **calculated EV** to Yahoo’s `enterpriseValue` in `info` when available.


In [43]:
ev_info = _safe_float(info.get("enterpriseValue"))

if not (np.isnan(market_cap) or np.isnan(total_debt) or np.isnan(cash_for_ev)):
    ev_calc = market_cap + total_debt - cash_for_ev
else:
    ev_calc = np.nan

ev = ev_info if not np.isnan(ev_info) else ev_calc

print("Enterprise value build:")
print(f"  Market cap:      {_fmt_usd(market_cap)}")
print(f"+ Total debt:      {_fmt_usd(total_debt)}")
print(f"- Cash:            {_fmt_usd(cash_for_ev)}")
print(f"= EV (calculated): {_fmt_usd(ev_calc)}")
print()
print(f"EV (Yahoo info):      {_fmt_usd(ev_info)}")
print(f"EV (used downstream): {_fmt_usd(ev)}")

total_assets = _latest_point(q_bs, ("total assets",))
print(f"\nTotal assets (LQ):    {_fmt_usd(total_assets)}")


Enterprise value build:
  Market cap:      $1,664,912,326,656  ($1.665 trillion)
+ Total debt:      $15,890,000,000  ($15.890 billion)
- Cash:            $16,603,000,000  ($16.603 billion)
= EV (calculated): $1,664,199,326,656  ($1.664 trillion)

EV (Yahoo info):      $1,636,745,347,072  ($1.637 trillion)
EV (used downstream): $1,636,745,347,072  ($1.637 trillion)

Total assets (LQ):    $143,724,000,000  ($143.724 billion)


In [44]:
trailing_eps = _safe_float(info.get("trailingEps"))
forward_eps = _safe_float(info.get("forwardEps"))
forward_pe = _safe_float(info.get("forwardPE"))
trailing_pe = _safe_float(info.get("trailingPE"))
peg_info = _safe_float(info.get("pegRatio"))
div_yield_info = info.get("dividendYield")
if div_yield_info is not None:
    div_yield_info = float(div_yield_info)
    # Yahoo often stores dividend yield as decimal (e.g. 0.02); if > 1, treat as percent
    if div_yield_info is not None and div_yield_info > 1.0:
        div_yield_info = div_yield_info / 100.0

book_value_ps = _safe_float(info.get("bookValue"))

# TTM flows
cfo_ttm = _ttm_sum(q_cf, ("operating cash flow",))
if np.isnan(cfo_ttm):
    cfo_ttm = _ttm_sum(q_cf, ("cash from operations",))
capex_ttm = _ttm_sum(q_cf, ("capital expenditure",))
if np.isnan(capex_ttm):
    capex_ttm = _ttm_sum(q_cf, ("capital", "expenditure"))

rev_ttm = _ttm_sum(q_fin, ("total revenue",))
ni_ttm = _ttm_sum(q_fin, ("net income",))
if np.isnan(ni_ttm):
    ni_ttm = _ttm_sum(q_fin, ("net", "income"))
ebitda_ttm = _ebitda_ttm(q_fin, q_cf)


total_equity = _latest_point(q_bs, ("stockholders", "equity"))
if np.isnan(total_equity):
    total_equity = _latest_point(q_bs, ("shareholders", "equity"))
if np.isnan(total_equity):
    total_equity = _latest_point(q_bs, ("total equity",))



## 4. Value factor row

Metrics match the cheat sheet in the chapter notes: yields are **ratios** (not percent) unless noted. `forward_pe_vs_sector` uses **your** `FORWARD_PE_PEERS` list only.

In [45]:
def _ratio(num: float, den: float) -> float:
    if np.isnan(num) or np.isnan(den) or den == 0:
        return np.nan
    return float(num / den)


def forward_pe_peer_gap(symbol: str, peers: Iterable[str]) -> float:
    peers = list(peers)
    if not peers:
        return np.nan
    t0 = yf.Ticker(symbol)
    f0 = _safe_float((t0.info or {}).get("forwardPE"))
    vals = []
    for p in peers:
        fp = _safe_float((yf.Ticker(p).info or {}).get("forwardPE"))
        if not np.isnan(fp):
            vals.append(fp)
    if np.isnan(f0) or not vals:
        return np.nan
    return float(f0 - np.mean(vals))


cash_flow_yield = _ratio(cfo_ttm, market_cap)
cash_flow_yield_alt = _ratio(cfo_ttm / shares, price) if not (
    np.isnan(cfo_ttm) or np.isnan(shares) or shares == 0 or np.isnan(price)
) else np.nan

fcf_yield = _ratio(fcf_ttm, market_cap)
den_ic = total_debt + total_equity
cfroic = _ratio(cfo_ttm, den_ic)
cfo_to_total_assets = _ratio(cfo_ttm, total_assets)
fcf_to_ev = _ratio(fcf_ttm, ev)
ebitda_to_ev = _ratio(ebitda_ttm, ev)

earnings_yield_trailing = _ratio(eps_ttm_approx, price)
if np.isnan(earnings_yield_trailing):
    earnings_yield_trailing = _ratio(ni_ttm, market_cap)

earnings_yield_forward = _ratio(forward_eps, price)

peg_ratio = peg_info
if np.isnan(peg_ratio) and not (np.isnan(forward_pe) or np.isnan(trailing_pe)):
    eg = _safe_float(info.get("earningsGrowth"))  # often fraction for 1y
    if not np.isnan(eg) and eg != 0:
        # If earningsGrowth is fraction (0.15 = 15%), PEG ~ PE / (100*g) with PE on forward basis is messy; use forwardPE / (g*100) if g is fraction
        g = eg * 100 if abs(eg) < 1.0 else eg
        peg_ratio = _ratio(forward_pe if not np.isnan(forward_pe) else trailing_pe, g)

forward_pe_vs_sector = forward_pe_peer_gap(SYMBOL, FORWARD_PE_PEERS)

sales_yield = _ratio(rev_ttm / shares, price) if not (
    np.isnan(rev_ttm) or np.isnan(shares) or shares == 0 or np.isnan(price)
) else np.nan

# Forward sales yield: Yahoo rarely gives consensus sales per share; leave NaN with documented limitation
sales_yield_forward = np.nan

book_value_yield = _ratio(book_value_ps, price)

dividend_yield = div_yield_info if div_yield_info is not None else np.nan
if np.isnan(dividend_yield):
    t12 = _safe_float(info.get("trailingAnnualDividendRate"))
    dividend_yield = _ratio(t12, price)

value_factors = pd.Series(
    {
        "symbol": SYMBOL,
        "price": price,
        "market_cap": market_cap,
        "enterprise_value": ev,
        "shares_outstanding": shares,
        "cfo_ttm": cfo_ttm,
        "capex_ttm": capex_ttm,
        "fcf_ttm": fcf_ttm,
        "revenue_ttm": rev_ttm,
        "net_income_ttm": ni_ttm,
        "ebitda_ttm": ebitda_ttm,
        "total_assets_lq": total_assets,
        "total_debt_lq": total_debt,
        "total_equity_lq": total_equity,
        "cash_lq": cash_bs,
        "cash_flow_yield": cash_flow_yield,
        "cash_flow_yield_ocf_per_share_over_price": cash_flow_yield_alt,
        "fcf_yield": fcf_yield,
        "cfroic": cfroic,
        "cfo_to_total_assets": cfo_to_total_assets,
        "fcf_to_ev": fcf_to_ev,
        "ebitda_to_ev": ebitda_to_ev,
        "earnings_yield_trailing": earnings_yield_trailing,
        "earnings_yield_forward": earnings_yield_forward,
        "peg_ratio": peg_ratio,
        "forward_pe_vs_sector": forward_pe_vs_sector,
        "sales_yield": sales_yield,
        "sales_yield_forward": sales_yield_forward,
        "book_value_yield": book_value_yield,
        "dividend_yield": dividend_yield,
    }
)

pd.DataFrame([value_factors]).T

,0
symbol,TSLA
price,443.3
market_cap,1664912326656.0
enterprise_value,1636745347072.0
shares_outstanding,3755723871.0
cfo_ttm,16528000000.0
capex_ttm,-9528000000.0
fcf_ttm,7000000000.0
revenue_ttm,97879000000.0
net_income_ttm,3862000000.0


## 5. Optional: Financial Modeling Prep and Polygon (skeleton)

**Polygon.io** (when subscribed) exposes fundamentals and reference data under their REST API; map the same ratios as above from vendor fields or by rebuilding TTM from reported financials—use their docs for ticker identifiers and filing timestamps if you need cleaner alignment with price bars.

Typical **FMP** mappings (check current docs for exact paths and parameters):

- **Statements (stable API)**: `/stable/income-statement`, `balance-sheet-statement`, `cash-flow-statement` with `symbol` + `period=quarter` (see FMP docs). Legacy `/api/v3/...` paths often return **403** for newer keys.
- **Ratios / key metrics**: `/stable/key-metrics-ttm?symbol=...` (and related `key-metrics`, `ratios` stable routes) for precomputed yields and valuation multiples.
- **Analyst estimates**: `analyst-estimates` or company estimate endpoints for **forward EPS / revenue** and growth rates used in PEG or forward sales yield.

Set `FMP_API_KEY` in the environment and re-run the cell below to hit a sample endpoint (disabled when the key is missing).

In [46]:
# import requests
# from IPython.display import Markdown, display


# def fmp_series_to_markdown(s: pd.Series, title: str, max_rows: int = 40) -> str:
#     """Render the first rows of an FMP metrics Series as a GitHub-flavored markdown table."""
#     lines = [f"### {title}", "", "| Field | Value |", "| --- | ---: |"]
#     for k, v in s.head(max_rows).items():
#         key = str(k).replace("|", "\\|").replace("`", "")
#         if isinstance(v, (bool, np.bool_)):
#             val = str(bool(v))
#         elif isinstance(v, (int, np.integer)):
#             val = f"{int(v):,}"
#         elif isinstance(v, (float, np.floating)):
#             val = f"{float(v):,.6g}"
#         elif v is None or (isinstance(v, float) and np.isnan(v)):
#             val = ""
#         else:
#             val = str(v).replace("|", "\\|")
#         lines.append(f"| `{key}` | {val} |")
#     return "\n".join(lines)


# def fetch_fmp_key_metrics_ttm(symbol: str, api_key: str) -> Optional[pd.Series]:
#     """Map FMP key-metrics TTM JSON into a flat Series.

#     FMP documents the **stable** base path. Legacy ``/api/v3/key-metrics-ttm/{symbol}`` often returns **403**
#     for newer keys, so we try ``/stable`` first, then legacy.
#     """
#     if not api_key:
#         return None
#     sym = str(symbol).strip().upper()
#     candidates = [
#         (
#             "https://financialmodelingprep.com/stable/key-metrics-ttm",
#             {"symbol": sym, "apikey": api_key},
#         ),
#         (
#             f"https://financialmodelingprep.com/api/v3/key-metrics-ttm/{sym}",
#             {"apikey": api_key},
#         ),
#     ]
#     for url, params in candidates:
#         try:
#             r = requests.get(url, params=params, timeout=30)
#         except requests.RequestException as exc:
#             print(f"FMP request failed ({url}): {exc}")
#             continue
#         if r.status_code != 200:
#             snippet = (r.text or "")[:300].replace("\n", " ")
#             print(f"FMP HTTP {r.status_code} from {url} — {snippet}")
#             continue
#         try:
#             data = r.json()
#         except ValueError:
#             print("FMP: response was not valid JSON")
#             continue
#         if not data:
#             return None
#         row = data[0] if isinstance(data, list) else data
#         return pd.Series(row)
#     print(
#         "FMP: could not load key-metrics-ttm. If both stable and legacy fail, verify the key, "
#         "your subscription tier, and FMP’s current docs (search: stable key-metrics-ttm)."
#     )
#     return None


# if FMP_API_KEY:
#     fmp_row = fetch_fmp_key_metrics_ttm(SYMBOL, FMP_API_KEY)
#     if fmp_row is not None:
#         display(Markdown(fmp_series_to_markdown(fmp_row, f"FMP key-metrics TTM — {SYMBOL}")))
#     else:
#         print("No FMP key-metrics-ttm row returned.")
# else:
#     print("Skipping FMP: set environment variable FMP_API_KEY to enable.")

### Example: FMP `key-metrics-ttm` (illustrative TSLA snapshot)

The code cell above prints the **same** fields as a rendered table when you run it with a valid `FMP_API_KEY`. Static copy of a sample pull:

| Field | Value |
| --- | ---: |
| `symbol` | TSLA |
| `marketCap` | 1,664,910,676,000 |
| `enterpriseValueTTM` | 1,657,536,676,000 |
| `evToSalesTTM` | 16.934549 |
| `evToOperatingCashFlowTTM` | 100.286585 |
| `evToFreeCashFlowTTM` | 236.790954 |
| `evToEBITDATTM` | 158.207185 |
| `netDebtToEBITDATTM` | -0.703827 |
| `currentRatioTTM` | 2.043119 |
| `incomeQualityTTM` | 4.26419 |
| `grahamNumberTTM` | 26.483458 |
| `grahamNetNetTTM` | -1.252242 |
| `taxBurdenTTM` | 0.712893 |
| `interestBurdenTTM` | 0.941309 |
| `workingCapitalTTM` | 35,610,000,000 |
| `investedCapitalTTM` | 91,555,000,000 |
| `returnOnAssetsTTM` | 0.026968 |
| `operatingReturnOnAssetsTTM` | 0.034788 |
| `returnOnTangibleAssetsTTM` | 0.026968 |
| `returnOnEquityTTM` | 0.047921 |
